<a href="https://colab.research.google.com/github/canurdon/SnowLine/blob/main/sentinel2_snow_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [75]:
import ee
import geemap
import xml.etree.ElementTree as ET
import datetime
import math
dem = ee.Image('USGS/SRTMGL1_003')


ee.Authenticate()
ee.Initialize(project='snowline-491111')

In [76]:
!pip install earthengine-api geemap -q

In [77]:
#create Tantalus bounding box
aoi = ee.Geometry.Rectangle([-123.38, 49.75, -123.10, 49.95])

#Verify area loaded correctly
print('area of interest defined')
print('Bounds:', aoi.bounds().getInfo())

area of interest defined
Bounds: {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-123.38000000000001, 49.74999999999998], [-123.10000000000001, 49.74999999999998], [-123.10000000000001, 49.95008424772833], [-123.38000000000001, 49.95008424772833], [-123.38000000000001, 49.74999999999998]]]}


In [78]:
# Dynamic date range — most recent 60 days
end = datetime.datetime.now()
start = end - datetime.timedelta(days=60)

end_str = end.strftime('%Y-%m-%d')
start_str = start.strftime('%Y-%m-%d')

print(f'Searching: {start_str} to {end_str}')

s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(aoi) \
    .filterDate(start_str, end_str) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
    .sort('system:time_start', False)

image = s2.first()

info = image.getInfo()
print('Image ID:', info['id'])
print('Date:', ee.Date(image.get('system:time_start')).format('YYYY-MM-dd').getInfo())
print('Cloud cover:', image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo(), '%')

Searching: 2026-01-25 to 2026-03-26
Image ID: COPERNICUS/S2_SR_HARMONIZED/20260308T192139_20260308T192314_T10UDA
Date: 2026-03-08
Cloud cover: 29.023671 %


In [79]:
# Select the bands we need
# Band 3 = Green, Band 11 =SWIR
green = image.select('B3')
swir = image.select('B11')

# Compute NDSI
# Formula: (Green - SWIR) / (Green + SWIR)
ndsi = image.normalizedDifference(['B3', 'B11']).rename('NDSI')

#Threshhold at 0.55 - standard snow detection cutoff
snow_mask = ndsi.gt(0.55)

# Visualise on an interactive map
map1 = geemap.Map()
map1.centerObject(aoi, zoom=11)

# Add layers
map1.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour'
)
map1.addLayer(
    ndsi,
    {'min': -1, 'max': 1, 'palette': ['black', 'white']},
    'NDSI'
)
map1.addLayer(
    snow_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow mask'
)

map1


Map(center=[49.85001533216063, -123.2400000000015], controls=(WidgetControl(options=['position', 'transparent_…

## Create cloud mask
Avoids classifying cloud as snow.

In [80]:
# SCL identifies the type of terrain represented by each pixel
# Numbers we want to mask out:
# 3 – cloud shadow; 8 – cloud medium probability; 9 – cloud high probability; 10 – thin cirrus

# create variable for SCL
scl = image.select('SCL')

# create cloud mask: keep = 1; mask out = 0
cloud_mask = scl.neq(3) \
.And(scl.neq(8)) \
.And(scl.neq(9)) \
.And(scl.neq(10))

# Apply cloud mask to NDSI
ndsi_masked = ndsi.updateMask(cloud_mask)
snow_mask_masked = ndsi_masked.gt(0.45)

# Add to map
map2 = geemap.Map()
map2.centerObject(aoi, zoom=11)

map2.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour'
)
map2.addLayer(
    snow_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow mask unmasked'
)
map2.addLayer(
    snow_mask_masked,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow mask masked'
)
map2.addLayer(
    cloud_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', 'ffffff']},
    'Cloud mask'
)

map2

Map(center=[49.85001533216063, -123.2400000000015], controls=(WidgetControl(options=['position', 'transparent_…

In [81]:
# SCL-based snow mask — ESA's own classification
# Value 11 = snow/ice in the Scene Classification Layer
scl_full = image.select('SCL')
scl_snow_mask = scl_full.eq(11)

# Compare against NDSI mask
# Four possible outcomes per pixel:
# Both agree snow     = true positive
# Both agree no snow  = true negative
# NDSI says snow, SCL says no = false positive (rock falsely detected)
# SCL says snow, NDSI says no = false negative (snow missed)

true_positive  = snow_mask.And(scl_snow_mask)
true_negative  = snow_mask.Not().And(scl_snow_mask.Not())
false_positive = snow_mask.And(scl_snow_mask.Not())
false_negative = snow_mask.Not().And(scl_snow_mask)

# Count pixels in each category over AOI
def count_pixels(img):
    return img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=20,
        maxPixels=1e9
    ).values().get(0)

tp = count_pixels(true_positive).getInfo()
tn = count_pixels(true_negative).getInfo()
fp = count_pixels(false_positive).getInfo()
fn = count_pixels(false_negative).getInfo()

total = tp + tn + fp + fn

print(f'True positives  (both agree snow):     {tp:>8.0f} pixels ({tp/total*100:.1f}%)')
print(f'True negatives  (both agree no snow):  {tn:>8.0f} pixels ({tn/total*100:.1f}%)')
print(f'False positives (NDSI snow, SCL clear):{fp:>8.0f} pixels ({fp/total*100:.1f}%)')
print(f'False negatives (SCL snow, NDSI clear):{fn:>8.0f} pixels ({fn/total*100:.1f}%)')
print()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
accuracy  = (tp + tn) / total

print(f'Precision: {precision:.3f}  (of pixels NDSI calls snow, how many does SCL agree?)')
print(f'Recall:    {recall:.3f}  (of pixels SCL calls snow, how many does NDSI catch?)')
print(f'Accuracy:  {accuracy:.3f}  (overall agreement)')

# Visualise disagreements on map
map5 = geemap.Map()
map5.centerObject(aoi, zoom=11)

map5.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour'
)
map5.addLayer(
    true_positive,
    {'min': 0, 'max': 1, 'palette': ['000000', '0000ff']},
    'True positive (both agree snow)'
)
map5.addLayer(
    false_positive,
    {'min': 0, 'max': 1, 'palette': ['000000', 'ff0000']},
    'False positive (NDSI says snow, SCL disagrees)'
)
map5.addLayer(
    false_negative,
    {'min': 0, 'max': 1, 'palette': ['000000', '00ff00']},
    'False negative (SCL says snow, NDSI misses)'
)
map5.addLayer(
    scl_snow_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', '00ffff']},
    'SCL snow only'
)

map5

True positives  (both agree snow):       182646 pixels (16.3%)
True negatives  (both agree no snow):    737476 pixels (65.9%)
False positives (NDSI snow, SCL clear):  167352 pixels (15.0%)
False negatives (SCL snow, NDSI clear):   31522 pixels (2.8%)

Precision: 0.522  (of pixels NDSI calls snow, how many does SCL agree?)
Recall:    0.853  (of pixels SCL calls snow, how many does NDSI catch?)
Accuracy:  0.822  (overall agreement)


Map(center=[49.85001533216063, -123.2400000000015], controls=(WidgetControl(options=['position', 'transparent_…

# Multiple Image Compilation
Using a compilation of images to get an mosaic that is resistant to cloud cover.


In [82]:
# All thresholds in one place. Adjust per region / season.

# Snow detection
NDSI_SNOW_THRESHOLD = 0.4       # standard snow/not-snow cutoff
NDSI_STRICT_THRESHOLD = 0.6     # high-confidence snow for snowline derivation
B3_BRIGHTNESS_FLOOR = 1500      # green band reflectance floor — rejects dark NDSI artifacts

# Snowline
SNOWLINE_BUFFER_M = 200         # extend snowline downward by this much (conservative)
SNOWLINE_PERCENTILE = 10        # percentile of confident-snow elevations = snowline

# Staleness
STALENESS_FRESH = 3             # days — pixel considered fresh
STALENESS_MODERATE = 7          # days — pixel considered moderate (>7 = stale)

# Route sampling
ROUTE_BUFFER_M = 30             # corridor width for sampling
MIN_SEGMENT_M = 200             # ignore snow patches shorter than this
MAX_GAP_M = 50                  # merge clear gaps shorter than this

In [83]:
# DEM is used post-composite for dynamic snowline derivation.
# No static elevation filter — the snowline adapts to conditions.

dem = ee.Image('USGS/SRTMGL1_003').clip(aoi)

print('DEM loaded — snowline will be derived dynamically after compositing')


DEM loaded — snowline will be derived dynamically after compositing


In [84]:
end_date = ee.Date(datetime.datetime.now())
start_date = end_date.advance(-60, 'day')

collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(aoi) \
    .filterDate(start_date, end_date)


def process_image(image):
    """Per-image processing: mask clouds/water, compute NDSI, track timestamp.

    Only the continuous NDSI value flows through the composite — no binary
    thresholding here. This preserves the full signal so post-composite
    classification can apply the snowline gate and produce confidence levels,
    not just snow/not-snow.
    """
    scl = image.select('SCL')

    cloud_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    water_mask = scl.neq(6)
    valid_mask = cloud_mask.And(water_mask)

    ndsi = image.normalizedDifference(['B3', 'B11']).rename('NDSI')
    ndsi_masked = ndsi.updateMask(valid_mask)

    timestamp = image.metadata('system:time_start').rename('timestamp').clip(aoi)

    return ndsi_masked.addBands(timestamp).clip(aoi) \
        .copyProperties(image, ['system:time_start'])


processed = collection.map(process_image)
composite = processed.qualityMosaic('timestamp').clip(aoi)

print('Composite bands:', composite.bandNames().getInfo())
# Expected: ['NDSI', 'timestamp']

Composite bands: ['NDSI', 'timestamp']


In [85]:
# This is the key step: derive the snowline from the composite itself,
# then re-classify using the snowline as a dynamic elevation gate.

ndsi_composite = composite.select('NDSI')

# Step 1: Find "definitely snow" pixels in the composite
# Strict NDSI + brightness floor = high-confidence snow
# We need the B3 brightness from a recent clear image for this.
# Use the most recent low-cloud image from the collection.
recent_clear = collection \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
    .sort('system:time_start', False) \
    .first()

# Cloud mask for the reference image
ref_scl = recent_clear.select('SCL')
ref_cloud_mask = ref_scl.neq(3).And(ref_scl.neq(8)).And(ref_scl.neq(9)).And(ref_scl.neq(10))
ref_water_mask = ref_scl.neq(6)
ref_ndsi = recent_clear.normalizedDifference(['B3', 'B11'])

strict_snow = ref_ndsi.gt(NDSI_STRICT_THRESHOLD) \
    .And(recent_clear.select('B3').gt(B3_BRIGHTNESS_FLOOR)) \
    .And(ref_cloud_mask) \
    .And(ref_water_mask)

# Step 2: Derive snowline = 10th percentile elevation of confident snow pixels
snowline_elev = dem.updateMask(strict_snow).reduceRegion(
    reducer=ee.Reducer.percentile([SNOWLINE_PERCENTILE]),
    geometry=aoi,
    scale=100,
    maxPixels=1e9
).get('elevation')

# Buffer downward for safety margin
snowline_buffered = ee.Number(snowline_elev).subtract(SNOWLINE_BUFFER_M)

print('Derived snowline:', snowline_elev.getInfo(), 'm')
print('Buffered snowline:', snowline_buffered.getInfo(), 'm')

# Step 3: Apply snowline to composite
above_snowline = dem.gte(snowline_buffered)
snow_binary = ndsi_composite.gt(NDSI_SNOW_THRESHOLD)
snow_confident = snow_binary.And(above_snowline)

# Step 4: Staleness
now_ms = ee.Date(datetime.datetime.now()).millis()
days_ago = composite.select('timestamp') \
    .subtract(now_ms).abs().divide(86400000).rename('days_ago')

# Step 5: Staleness classification band
staleness_class = days_ago.where(days_ago.lte(STALENESS_FRESH), 1) \
    .where(days_ago.gt(STALENESS_FRESH).And(days_ago.lte(STALENESS_MODERATE)), 2) \
    .where(days_ago.gt(STALENESS_MODERATE), 3) \
    .rename('staleness')

# ── Package final output ─────────────────────────────────────────────────
# snow = binary gate (NDSI > threshold AND above snowline)
# NDSI = continuous value — drives confidence display in the app
# above_snowline = elevation gate
# days_ago = pixel staleness in days
# staleness = classified staleness (1=fresh, 2=moderate, 3=stale)
output = snow_confident.rename('snow') \
    .addBands(ndsi_composite) \
    .addBands(above_snowline.rename('above_snowline')) \
    .addBands(days_ago) \
    .addBands(staleness_class)

print('\nFinal output bands:', output.bandNames().getInfo())
# Expected: ['snow', 'NDSI', 'above_snowline', 'days_ago', 'staleness']

Derived snowline: 1043.3560413589364 m
Buffered snowline: 843.3560413589364 m

Final output bands: ['snow', 'NDSI', 'above_snowline', 'days_ago', 'staleness']


In [86]:
map3 = geemap.Map()
map3.centerObject(aoi, zoom=11)

# True colour from most recent clear image for context
map3.addLayer(
    recent_clear,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour (recent clear)'
)

# Snow: final confident classification
map3.addLayer(
    output.select('snow'),
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow (confident)'
)

# NDSI intensity gradient
map3.addLayer(
    output.select('NDSI'),
    {'min': 0.2, 'max': 1, 'palette': ['2c1654', '4a90d9', '00aaff', 'e0f7ff']},
    'NDSI intensity'
)

# Snowline elevation contour — show where the dynamic cutoff sits
snowline_contour = dem.gte(snowline_buffered).And(dem.lt(snowline_buffered.add(50)))
map3.addLayer(
    snowline_contour.selfMask(),
    {'palette': ['ff6600']},
    'Snowline contour'
)

# Pixels below snowline that NDSI thinks are snow — the ones we're rejecting
rejected = snow_binary.And(above_snowline.Not())
map3.addLayer(
    rejected.selfMask(),
    {'palette': ['ff0000']},
    'Rejected (below snowline)'
)

# Staleness
map3.addLayer(
    output.select('days_ago'),
    {'min': 0, 'max': 60, 'palette': ['00ff00', 'ffff00', 'ff0000']},
    'Pixel age (days)'
)

# Elevation
map3.addLayer(
    dem,
    {'min': 0, 'max': 2600, 'palette': ['000000', '555555', 'aaaaaa', 'ffffff']},
    'Elevation (SRTM)'
)

map3


Map(center=[49.85001533216063, -123.2400000000015], controls=(WidgetControl(options=['position', 'transparent_…

# GPS Track Functionality
These cells allow the user to upload a GPS track and analyse the route against the terrain layers.

In [87]:
# Parse GPX file
def parse_gpx(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()

    # Handle GPX namespace
    ns = {'gpx': 'http://www.topografix.com/GPX/1/1'}

    coords = []
    elevations = []

    for trkpt in root.findall('.//gpx:trkpt', ns):
        lat = float(trkpt.get('lat'))
        lon = float(trkpt.get('lon'))
        ele = trkpt.find('gpx:ele', ns)
        elev = float(ele.text) if ele is not None else 0

        coords.append([lon, lat])
        elevations.append(elev)

    return coords, elevations

# Load the route
coords, elevations = parse_gpx('/content/tantalusFKT_2024.gpx')

print(f'Track points: {len(coords)}')
print(f'Start: {coords[0]}')
print(f'End: {coords[-1]}')
print(f'Min elevation: {min(elevations)}m')
print(f'Max elevation: {max(elevations)}m')

Track points: 21699
Start: [-123.3229366, 49.9105733]
End: [-123.2026349, 49.79482]
Min elevation: 101.0m
Max elevation: 2601.0m


In [89]:
# Simplify route — take every 50th point
# 21699 / 50 = ~434 points, plenty for 10m resolution
simplified = coords[::50]
print(f'Simplified to {len(simplified)} points')

# Create Earth Engine line geometry
route = ee.Geometry.LineString(simplified)

# Display on map with new AOI
map4 = geemap.Map()
map4.centerObject(route, zoom=11)

# Add snow composite
map4.addLayer(
    output.select('snow'),
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow composite'
)

# Add route
map4.addLayer(
    ee.Image().paint(route, 1, 2),
    {'palette': ['ff0000']},
    'Tantalus Traverse'
)

map4

Simplified to 434 points


Map(center=[49.837876508085344, -123.31442795002913], controls=(WidgetControl(options=['position', 'transparen…

In [90]:
route_buffer = route.buffer(ROUTE_BUFFER_M)

# Sample from the final output — all bands at once
sampled = output.select(['snow', 'NDSI', 'above_snowline', 'days_ago']) \
    .sample(
        region=route_buffer,
        scale=10,
        numPixels=500,
        geometries=True
    )

# Counts
total = sampled.size().getInfo()
snow_count = sampled.filter(ee.Filter.eq('snow', 1)).size().getInfo()
clear_count = total - snow_count

print(f'Total sampled points: {total}')
print(f'Snow points: {snow_count}')
print(f'Clear points: {clear_count}')
print(f'Snow coverage: {round(snow_count/total*100, 1)}%')

# Detailed stats
features = sampled.getInfo()['features']

ndsi_values = [f['properties']['NDSI'] for f in features
               if f['properties']['snow'] == 1 and f['properties']['NDSI'] is not None]
below_snowline_hits = sum(1 for f in features
                          if f['properties'].get('above_snowline') == 0
                          and f['properties']['NDSI'] is not None
                          and f['properties']['NDSI'] > NDSI_SNOW_THRESHOLD)

if ndsi_values:
    print(f'\nSnow pixel NDSI — mean: {sum(ndsi_values)/len(ndsi_values):.3f}, '
          f'min: {min(ndsi_values):.3f}, max: {max(ndsi_values):.3f}')
    print(f'Rejected by snowline: {below_snowline_hits} pixels had NDSI > {NDSI_SNOW_THRESHOLD} '
          f'but were below the derived snowline')


Total sampled points: 499
Snow points: 362
Clear points: 137
Snow coverage: 72.5%

Snow pixel NDSI — mean: 0.730, min: 0.416, max: 0.947
Rejected by snowline: 24 pixels had NDSI > 0.4 but were below the derived snowline


In [101]:
points = []
for f in features:
    coords = f['geometry']['coordinates']
    props = f['properties']

    points.append({
        'lon': coords[0],
        'lat': coords[1],
        'snow': props['snow'],
        'ndsi': props.get('NDSI'),
        'above_snowline': props.get('above_snowline'),
        'days_ago': round(props.get('days_ago'), 1) if props.get('days_ago') is not None else None,
    })

print(f'Extracted {len(points)} points')
print(f'Sample: {points[0]}')

Extracted 499 points
Sample: {'lon': -123.35247499735075, 'lat': 49.89418792966138, 'snow': 1, 'ndsi': 0.5133004784584045, 'above_snowline': 1, 'days_ago': 0.7}


In [102]:
def haversine(coord1, coord2):
    """Distance in metres between two [lon, lat] points"""
    R = 6371000
    lat1, lat2 = math.radians(coord1[1]), math.radians(coord2[1])
    dlat = math.radians(coord2[1] - coord1[1])
    dlon = math.radians(coord2[0] - coord1[0])
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# Build list of points with cumulative distance along route
# Use original simplified coords as the route backbone
route_with_dist = []
cumulative_dist = 0
for i, coord in enumerate(simplified):
    if i > 0:
        cumulative_dist += haversine(simplified[i-1], coord)
    route_with_dist.append({
        'lon': coord[0],
        'lat': coord[1],
        'dist_m': cumulative_dist
    })

total_route_km = cumulative_dist / 1000
print(f'Total route length: {total_route_km:.1f} km')

# For each sampled point, find nearest route point and get its distance
def nearest_route_dist(point, route):
    min_dist = float('inf')
    nearest = None
    for rp in route:
        d = haversine([point['lon'], point['lat']], [rp['lon'], rp['lat']])
        if d < min_dist:
            min_dist = d
            nearest = rp
    return nearest['dist_m']

# Tag each sample point with distance along route
print('Calculating distances along route...')
for p in points:
    p['route_dist_m'] = nearest_route_dist(p, route_with_dist)

# Sort by distance along route
points.sort(key=lambda x: x['route_dist_m'])
print('Done')


Total route length: 29.5 km
Calculating distances along route...
Done


In [105]:
segments_raw = []
current_state = points[0]['snow']
segment_start = points[0]
segment_points = [points[0]]

for p in points[1:]:
    if p['snow'] != current_state:
        dist_m = segment_points[-1]['route_dist_m'] - segment_start['route_dist_m']
        ndsi_vals = [pt['ndsi'] for pt in segment_points if pt['ndsi'] is not None]
        days_vals = [pt['days_ago'] for pt in segment_points if pt['days_ago'] is not None]

        segments_raw.append({
            'type': 'snow' if current_state == 1 else 'clear',
            'start_m': segment_start['route_dist_m'],
            'end_m': segment_points[-1]['route_dist_m'],
            'length_m': dist_m,
            'points': segment_points,
            'mean_ndsi': sum(ndsi_vals) / max(1, len(ndsi_vals)),
            'max_days_ago': max(days_vals) if days_vals else None,
        })
        current_state = p['snow']
        segment_start = p
        segment_points = [p]
    else:
        segment_points.append(p)

# Last segment
ndsi_vals = [pt['ndsi'] for pt in segment_points if pt['ndsi'] is not None]
days_vals = [pt['days_ago'] for pt in segment_points if pt['days_ago'] is not None]
segments_raw.append({
    'type': 'snow' if current_state == 1 else 'clear',
    'start_m': segment_start['route_dist_m'],
    'end_m': segment_points[-1]['route_dist_m'],
    'length_m': segment_points[-1]['route_dist_m'] - segment_start['route_dist_m'],
    'points': segment_points,
    'mean_ndsi': sum(ndsi_vals) / max(1, len(ndsi_vals)),
    'max_days_ago': max(days_vals) if days_vals else None,
})

# Gap merging
segments_merged = [segments_raw[0]]
for seg in segments_raw[1:]:
    prev = segments_merged[-1]
    if (prev['type'] == 'clear'
            and prev['length_m'] < MAX_GAP_M
            and len(segments_merged) >= 2
            and segments_merged[-2]['type'] == 'snow'
            and seg['type'] == 'snow'):
        snow_before = segments_merged[-2]
        all_points = snow_before['points'] + prev['points'] + seg['points']
        ndsi_all = [pt['ndsi'] for pt in all_points if pt['ndsi'] is not None]
        days_all = [pt['days_ago'] for pt in all_points if pt['days_ago'] is not None]
        snow_before['end_m'] = seg['end_m']
        snow_before['length_m'] = seg['end_m'] - snow_before['start_m']
        snow_before['points'] = all_points
        snow_before['mean_ndsi'] = sum(ndsi_all) / max(1, len(ndsi_all))
        snow_before['max_days_ago'] = max(days_all) if days_all else None
        segments_merged.pop()
    else:
        segments_merged.append(seg)

# Length filter
segments_final = [s for s in segments_merged if s['length_m'] >= MIN_SEGMENT_M]

# Report
snow_segs = [s for s in segments_final if s['type'] == 'snow']
print(f'\nSnow crossings: {len(snow_segs)}')
print(f'({len(segments_raw)} raw → {len(segments_merged)} merged → {len(segments_final)} final)\n')

for i, seg in enumerate(snow_segs):
    staleness = f"{seg['max_days_ago']:.0f}d old" if seg['max_days_ago'] else "age unknown"
    ndsi_str = f"NDSI {seg['mean_ndsi']:.2f}"
    confidence = "strong" if seg['mean_ndsi'] > 0.6 else "moderate" if seg['mean_ndsi'] > 0.45 else "weak"

    print(f"  Crossing {i+1}: {seg['start_m']/1000:.1f}–{seg['end_m']/1000:.1f} km "
          f"({seg['length_m']:.0f}m) | {ndsi_str} ({confidence}) | {staleness}")



Snow crossings: 2
(15 raw → 5 merged → 4 final)

  Crossing 1: 4.2–4.9 km (687m) | NDSI 0.53 (moderate) | 1d old
  Crossing 2: 5.0–26.2 km (21163m) | NDSI 0.73 (strong) | 1d old
